In [1]:
import sqlite3
import pandas as pd

df = pd.read_csv("../data/telco_scored.csv")

conn = sqlite3.connect("../data/telco.db")
df.to_sql("customers", conn, if_exists="replace", index=False)

print(pd.read_sql("SELECT COUNT(*) AS total_customers FROM customers", conn))
print()
print(pd.read_sql("""
    SELECT customerID, Contract, tenure, MonthlyCharges, Churn, risk_category
    FROM customers
    LIMIT 5
""", conn))

   total_customers
0             7043

   customerID        Contract  tenure  MonthlyCharges  Churn risk_category
0  7590-VHVEG  Month-to-month       1           29.85      0          High
1  5575-GNVDE        One year      34           56.95      0           Low
2  3668-QPYBK  Month-to-month       2           53.85      1        Medium
3  7795-CFOCW        One year      45           42.30      0           Low
4  9237-HQITU  Month-to-month       2           70.70      1          High


In [3]:
print("QUERY 1: Overall churn rate")
print(pd.read_sql("""
    SELECT COUNT(*) AS total_customers,
           SUM(Churn) AS churned,
           ROUND(100.0 * SUM(Churn) / COUNT(*), 2) AS churn_rate_pct
    FROM customers
""", conn))

print()
print("QUERY 2: Customers paying more than 100 a month")
print(pd.read_sql("""
    SELECT COUNT(*) AS customers,
           ROUND(100.0 * SUM(Churn) / COUNT(*), 1) AS churn_rate_pct
    FROM customers
    WHERE MonthlyCharges > 100
""", conn))

print()
print("QUERY 3: Churn by contract type")
print(pd.read_sql("""
    SELECT Contract,
           COUNT(*) AS customers,
           ROUND(100.0 * SUM(Churn) / COUNT(*), 1) AS churn_rate_pct
    FROM customers
    GROUP BY Contract
    ORDER BY churn_rate_pct DESC
""", conn))

QUERY 1: Overall churn rate
   total_customers  churned  churn_rate_pct
0             7043     1869           26.54

QUERY 2: Customers paying more than 100 a month
   customers  churn_rate_pct
0        902            28.0

QUERY 3: Churn by contract type
         Contract  customers  churn_rate_pct
0  Month-to-month       3875            42.7
1        One year       1473            11.3
2        Two year       1695             2.8


In [4]:
print("QUERY 4: Churn by tenure group (using CASE)")
print(pd.read_sql("""
    SELECT CASE
             WHEN tenure <= 12 THEN '1) 0-12 months'
             WHEN tenure <= 24 THEN '2) 13-24 months'
             WHEN tenure <= 48 THEN '3) 25-48 months'
             ELSE '4) 49-72 months'
           END AS tenure_group,
           COUNT(*) AS customers,
           ROUND(100.0 * SUM(Churn) / COUNT(*), 1) AS churn_rate_pct
    FROM customers
    GROUP BY tenure_group
    ORDER BY tenure_group
""", conn))

print()
print("QUERY 5: Average monthly spending by segment")
print(pd.read_sql("""
    SELECT Contract,
           ROUND(AVG(MonthlyCharges), 1) AS avg_monthly_charge
    FROM customers
    GROUP BY Contract
""", conn))
print(pd.read_sql("""
    SELECT InternetService,
           ROUND(AVG(MonthlyCharges), 1) AS avg_monthly_charge
    FROM customers
    GROUP BY InternetService
""", conn))

print()
print("QUERY 6: Customer groups with the highest churn")
print(pd.read_sql("""
    SELECT Contract, InternetService,
           COUNT(*) AS customers,
           ROUND(100.0 * SUM(Churn) / COUNT(*), 1) AS churn_rate_pct
    FROM customers
    GROUP BY Contract, InternetService
    HAVING COUNT(*) >= 100
    ORDER BY churn_rate_pct DESC
    LIMIT 5
""", conn))

QUERY 4: Churn by tenure group (using CASE)
      tenure_group  customers  churn_rate_pct
0   1) 0-12 months       2186            47.4
1  2) 13-24 months       1024            28.7
2  3) 25-48 months       1594            20.4
3  4) 49-72 months       2239             9.5

QUERY 5: Average monthly spending by segment
         Contract  avg_monthly_charge
0  Month-to-month                66.4
1        One year                65.0
2        Two year                60.8
  InternetService  avg_monthly_charge
0             DSL                58.1
1     Fiber optic                91.5
2              No                21.1

QUERY 6: Customer groups with the highest churn
         Contract InternetService  customers  churn_rate_pct
0  Month-to-month     Fiber optic       2128            54.6
1  Month-to-month             DSL       1223            32.2
2        One year     Fiber optic        539            19.3
3  Month-to-month              No        524            18.9
4        One year     

In [5]:
print("QUERY 7: Customers paying MORE than the average monthly charge (subquery)")
print(pd.read_sql("""
    SELECT COUNT(*) AS customers,
           ROUND(100.0 * SUM(Churn) / COUNT(*), 1) AS churn_rate_pct
    FROM customers
    WHERE MonthlyCharges > (SELECT AVG(MonthlyCharges) FROM customers)
""", conn))

print()
print("QUERY 8: Each contract's share of ALL churners (window function)")
print(pd.read_sql("""
    SELECT Contract,
           SUM(Churn) AS churned,
           ROUND(100.0 * SUM(Churn) / SUM(SUM(Churn)) OVER (), 1) AS pct_of_all_churners
    FROM customers
    GROUP BY Contract
    ORDER BY churned DESC
""", conn))

print()
print("QUERY 9: Top 3 highest-paying customers in each contract type")
print(pd.read_sql("""
    SELECT Contract, customerID, MonthlyCharges, Churn, rank_in_contract
    FROM (
        SELECT Contract, customerID, MonthlyCharges, Churn,
               ROW_NUMBER() OVER (PARTITION BY Contract
                                  ORDER BY MonthlyCharges DESC) AS rank_in_contract
        FROM customers
    )
    WHERE rank_in_contract <= 3
    ORDER BY Contract, rank_in_contract
""", conn))

QUERY 7: Customers paying MORE than the average monthly charge (subquery)
   customers  churn_rate_pct
0       3923            34.5

QUERY 8: Each contract's share of ALL churners (window function)
         Contract  churned  pct_of_all_churners
0  Month-to-month     1655                 88.6
1        One year      166                  8.9
2        Two year       48                  2.6

QUERY 9: Top 3 highest-paying customers in each contract type
         Contract  customerID  MonthlyCharges  Churn  rank_in_contract
0  Month-to-month  2302-ANTDP          117.45      1                 1
1  Month-to-month  8016-NCFVO          116.50      0                 2
2  Month-to-month  9659-QEQSY          115.65      0                 3
3        One year  5734-EJKXG          118.60      0                 1
4        One year  8199-ZLLSA          118.35      1                 2
5        One year  2889-FPWRM          117.80      1                 3
6        Two year  7569-NMZYQ          118.75     

In [6]:
actions = pd.DataFrame({
    "risk_category": ["Low", "Medium", "High"],
    "suggested_action": ["No action",
                         "Low-cost nudge (email or small offer)",
                         "Personal call or retention offer"]
})
actions.to_sql("risk_actions", conn, if_exists="replace", index=False)

print("QUERY 10: JOIN customers with the action table")
print(pd.read_sql("""
    SELECT c.risk_category,
           a.suggested_action,
           COUNT(*) AS customers,
           ROUND(100.0 * SUM(c.Churn) / COUNT(*), 1) AS churn_rate_pct
    FROM customers AS c
    INNER JOIN risk_actions AS a
        ON c.risk_category = a.risk_category
    GROUP BY c.risk_category, a.suggested_action
    ORDER BY churn_rate_pct DESC
""", conn))

QUERY 10: JOIN customers with the action table
  risk_category                       suggested_action  customers  \
0          High       Personal call or retention offer        969   
1        Medium  Low-cost nudge (email or small offer)       1697   
2           Low                              No action       4377   

   churn_rate_pct  
0            71.3  
1            44.1  
2             9.8  
